# 03 — Audio Processor

Generates scene narration audio and checks that the content duration plus the source card fits 80–100 seconds.

In [ ]:
import os, sys, subprocess
ROOT="/content/black-history-factory"
REPO_URL="https://github.com/jonbBla/black-history-factory.git"
if not os.path.exists(ROOT):
    subprocess.run(["git","clone",REPO_URL,ROOT], check=True)
sys.path.insert(0, ROOT)
subprocess.run([sys.executable,"-m","pip","install","-q","piper-tts"],check=True)
subprocess.run(["apt-get","update","-qq"],check=True); subprocess.run(["apt-get","install","-y","-qq","ffmpeg"],check=True)
from factory.drive import mount_drive,DrivePaths
from factory.config import Config
MYDRIVE=mount_drive(); paths=DrivePaths(os.path.join(MYDRIVE,"BLACK_HISTORY_FACTORY")); paths.ensure_tree(); config=Config.load(paths.root)
print(f"[SETUP] Drive: {paths.root}")


In [ ]:
from factory.audio_engine import load_piper_voice
PIPER_MODEL="/content/en_US-lessac-medium.onnx"
if not os.path.exists(PIPER_MODEL): raise FileNotFoundError("Upload your Piper .onnx voice model and set PIPER_MODEL")
voice=load_piper_voice(PIPER_MODEL)
print("[AUDIO] Piper voice loaded.")


In [ ]:
from factory.audio_engine import run
from factory.utils import read_json,write_json_atomic
from factory import status
def process_one():
    jobs=[]
    for jid in sorted(os.listdir(paths("02_JOBS"))):
        m=read_json(paths.manifest(jid),{}) or {}
        if m.get("status") in ("IMAGES_READY","AUDIO_REVIEW_REQUIRED"): jobs.append(jid)
    if not jobs:
        print("[AUDIO] No image-ready job."); return False
    jid=jobs[0]; scenes=read_json(paths.scenes(jid),[])
    def progress(n,total): status.set_processor(paths,"audio","running",jid,"narration",f"scene {n}/{total}",n,total)
    status.set_processor(paths,"audio","running",jid,"narration","starting",0,len(scenes))
    try:
        files,seconds=run(paths,jid,scenes,voice,config,progress)
        state=read_json(paths.state(jid,"audio"),{}) or {}; m=read_json(paths.manifest(jid),{}) or {}
        if state.get("status")=="AUDIO_READY":
            m.update(status="AUDIO_READY",audio_seconds=seconds); status.set_processor(paths,"audio","idle",jid,"ready",f"{seconds:.1f}s content",len(scenes),len(scenes)); print(f"[AUDIO] COMPLETE {jid} | {seconds:.1f}s")
        else:
            m.update(status="AUDIO_REVIEW_REQUIRED",audio_seconds=seconds); status.set_processor(paths,"audio","warning",jid,"duration_check",f"{seconds:.1f}s content outside target",len(scenes),len(scenes)); print(f"[AUDIO] REVIEW {jid} | {seconds:.1f}s content")
        write_json_atomic(paths.manifest(jid),m); return True
    except Exception as e:
        status.set_processor(paths,"audio","error",jid,"failed",str(e)); print(f"[AUDIO] ERROR {jid} | {e}"); return False
process_one()


In [ ]:
# Optional after the one-video test.
while process_one(): pass
